<a href="https://colab.research.google.com/github/dora-lai/fyp-aspect-financial-news/blob/main/3runs_LLM_results_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Load the three runs
print("Loading the 3 runs")
try:
    run1 = pd.read_csv("llm_run_1_merged.csv")
    run2 = pd.read_csv("llm_run_2_merged.csv")
    run3 = pd.read_csv("llm_run_3_merged.csv")
except FileNotFoundError as e:
    print(f"Error: {e}. Please make sure all 3 CSV files are uploaded to Colab.")
    raise

# Helper function to clean and evaluate a single run
def evaluate_run(df, run_name):
    # Clean data
    df = df.dropna(subset=['sentiment', 'llm_sentiment', 'aspect', 'llm_aspect']).copy()

    # Lowercase and strip for accurate comparison
    y_true_sent = df['sentiment'].astype(str).str.lower().str.strip()
    y_pred_sent = df['llm_sentiment'].astype(str).str.lower().str.strip()

    y_true_asp = df['aspect'].astype(str).str.lower().str.strip()
    y_pred_asp = df['llm_aspect'].astype(str).str.lower().str.strip()

    # Calculate Sentiment metrics
    sent_acc = accuracy_score(y_true_sent, y_pred_sent)
    _, _, f1_sent, _ = precision_recall_fscore_support(y_true_sent, y_pred_sent, average='macro', zero_division=0)

    # Calculate Aspect metrics
    asp_acc = accuracy_score(y_true_asp, y_pred_asp)
    _, _, f1_asp, _ = precision_recall_fscore_support(y_true_asp, y_pred_asp, average='macro', zero_division=0)

    print(f"\n--- {run_name} ---")
    print(f"Sentiment | Accuracy: {sent_acc:.4f} | Macro F1: {f1_sent:.4f}")
    print(f"Aspect    | Accuracy: {asp_acc:.4f} | Macro F1: {f1_asp:.4f}")

    return df

# 2. Evaluate each run individually
df1 = evaluate_run(run1, "Run 1")
df2 = evaluate_run(run2, "Run 2")
df3 = evaluate_run(run3, "Run 3")

# 3. Calculate Run-to-Run Variance (Consistency)
print("\n" + "="*50)
print("RUN-TO-RUN VARIANCE (CONSISTENCY)")
print("="*50)

# merge the three dataframes based on sentence and entity
# use lowercase entity names to ensure they match perfectly
for df in [df1, df2, df3]:
    df['entity_clean'] = df['llm_entity'].astype(str).str.lower().str.strip()

# Merge the three runs
merged = pd.merge(df1[['headline_id', 'entity_clean', 'llm_sentiment', 'llm_aspect']],
                  df2[['headline_id', 'entity_clean', 'llm_sentiment', 'llm_aspect']],
                  on=['headline_id', 'entity_clean'], suffixes=('_run1', '_run2'))

merged = pd.merge(merged,
                  df3[['headline_id', 'entity_clean', 'llm_sentiment', 'llm_aspect']],
                  on=['headline_id', 'entity_clean'])

merged.rename(columns={'llm_sentiment': 'llm_sentiment_run3', 'llm_aspect': 'llm_aspect_run3'}, inplace=True)

# Check if all 3 runs predicted the same sentiment
merged['sent_consistent'] = (merged['llm_sentiment_run1'] == merged['llm_sentiment_run2']) & \
                            (merged['llm_sentiment_run2'] == merged['llm_sentiment_run3'])

# Check if all 3 runs predicted the same aspect
merged['asp_consistent'] = (merged['llm_aspect_run1'] == merged['llm_aspect_run2']) & \
                           (merged['llm_aspect_run2'] == merged['llm_aspect_run3'])

sent_consistency_rate = merged['sent_consistent'].mean() * 100
asp_consistency_rate = merged['asp_consistent'].mean() * 100

print(f"Total aligned entity rows: {len(merged)}")
print(f"Sentiment Consistency: {sent_consistency_rate:.2f}%")
print(f"Aspect Consistency:    {asp_consistency_rate:.2f}%")

# 4. Show example of inconsistency
inconsistent = merged[~merged['sent_consistent']]
if len(inconsistent) > 0:
    print("\nExample of inconsistent sentiment prediction:")
    print(inconsistent[['headline_id', 'entity_clean', 'llm_sentiment_run1', 'llm_sentiment_run2', 'llm_sentiment_run3']].head(3).to_string())

Loading the 3 runs

--- Run 1 ---
Sentiment | Accuracy: 0.8667 | Macro F1: 0.8567
Aspect    | Accuracy: 0.8364 | Macro F1: 0.7696

--- Run 2 ---
Sentiment | Accuracy: 0.8667 | Macro F1: 0.8581
Aspect    | Accuracy: 0.8000 | Macro F1: 0.7320

--- Run 3 ---
Sentiment | Accuracy: 0.8727 | Macro F1: 0.8659
Aspect    | Accuracy: 0.8121 | Macro F1: 0.7815

RUN-TO-RUN VARIANCE (CONSISTENCY)
Total aligned entity rows: 164
Sentiment Consistency: 96.95%
Aspect Consistency:    91.46%

Example of inconsistent sentiment prediction:
    headline_id             entity_clean llm_sentiment_run1 llm_sentiment_run2 llm_sentiment_run3
19           23              ulta beauty           negative           negative            neutral
55           64           essex property           negative           negative            neutral
66           76  missouri american water           positive            neutral            neutral


In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support as prf
rows = []
for k in (1, 2, 3):
    d = pd.read_csv(f"llm_run_{k}_merged.csv").dropna(subset=['sentiment', 'sentence']).copy()
    m = d['llm_entity'].notna()           # gold rows where the LLM gave the same entity name
    for c in ['entity', 'aspect', 'sentiment', 'llm_entity', 'llm_aspect', 'llm_sentiment']:
        d[c] = d[c].astype(str).str.strip().str.lower()
    f = lambda y, p: prf(list(y), list(p), labels=['negative', 'neutral', 'positive'],
                         average='macro', zero_division=0)[2]
    rows.append({'run': k, 'n_all': len(d), 'n_matched': int(m.sum()),
                 'entity_acc': m.mean(),
                 'pair_all': ((d.aspect == d.llm_aspect) & (d.sentiment == d.llm_sentiment)).mean(),
                 'aspect_acc_all': (d.aspect == d.llm_aspect).mean(),
                 'aspect_acc_matched': (d.aspect[m] == d.llm_aspect[m]).mean(),
                 'sentF1_all': f(d.sentiment, d.llm_sentiment.where(m, 'missing')),
                 'sentF1_matched': f(d.sentiment[m], d.llm_sentiment[m])})
r = pd.DataFrame(rows).set_index('run')
pd.set_option('display.width', 200)
print(r.round(4)); print("\nmean\n", r.mean().round(4)); print("\nsd\n", r.std().round(4))

     n_all  n_matched  entity_acc  pair_all  aspect_acc_all  aspect_acc_matched  sentF1_all  sentF1_matched
run                                                                                                        
1      216        165      0.7639    0.5694          0.6389              0.8364      0.7445          0.8567
2      216        165      0.7639    0.5509          0.6111              0.8000      0.7431          0.8581
3      216        165      0.7639    0.5556          0.6204              0.8121      0.7495          0.8659

mean
 n_all                 216.0000
n_matched             165.0000
entity_acc              0.7639
pair_all                0.5586
aspect_acc_all          0.6235
aspect_acc_matched      0.8162
sentF1_all              0.7457
sentF1_matched          0.8602
dtype: float64

sd
 n_all                 0.0000
n_matched             0.0000
entity_acc            0.0000
pair_all              0.0096
aspect_acc_all        0.0141
aspect_acc_matched    0.0185
sentF1_all 

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import precision_recall_fscore_support as prf
ASP = ['earnings', 'management', 'other', 'regulation', 'supply chain', 'valuation']
runs = []
for k in (1, 2, 3):
    d = pd.read_csv(f"llm_run_{k}_merged.csv").dropna(subset=['sentiment', 'sentence']).copy()
    d['key'] = d['headline_id'].astype(int).astype(str) + '|' + d['entity'].astype(str).str.strip().str.lower()
    d['matched'] = d['llm_entity'].notna()
    for c in ['aspect', 'sentiment', 'llm_aspect', 'llm_sentiment']:
        d[c] = d[c].astype(str).str.strip().str.lower()
    d['p_asp'] = d['llm_aspect'].where(d['matched'], 'missing')
    d['p_sen'] = d['llm_sentiment'].where(d['matched'], 'missing')
    runs.append(d.set_index('key').sort_index())
assert all(r.index.equals(runs[0].index) for r in runs), "gold rows differ between runs"
n = len(runs[0]); print("gold rows:", n)

# (A) run agreement
asp = pd.concat([r['p_asp'] for r in runs], axis=1); sen = pd.concat([r['p_sen'] for r in runs], axis=1)
all_m = pd.concat([r['matched'] for r in runs], axis=1).all(axis=1)
same = lambda df: (df.nunique(axis=1) == 1)
print(f"\n(A) agreement, all {n} rows: aspect {same(asp).mean():.4f}  sentiment {same(sen).mean():.4f}")
print(f"    rows matched in all 3 runs (n={int(all_m.sum())}): aspect {same(asp[all_m]).mean():.4f}  sentiment {same(sen[all_m]).mean():.4f}")
print("    your reported values: aspect 0.9146, sentiment 0.9695")

# (B) aspect macro-F1
rows = []
for k, r in enumerate(runs, 1):
    m = r['matched']
    f = lambda y, p: prf(list(y), list(p), labels=ASP, average='macro', zero_division=0)[2]
    rows.append({'run': k, 'aspectF1_all': f(r['aspect'], r['p_asp']), 'aspectF1_matched': f(r['aspect'][m], r['llm_aspect'][m])})
b = pd.DataFrame(rows).set_index('run')
print("\n(B)\n", b.round(4)); print("mean\n", b.mean().round(4).to_string())
print("sd (n-1)\n", b.std(ddof=1).round(4).to_string()); print("sd (n)\n", b.std(ddof=0).round(4).to_string())
print("    your reported value: 0.7610 +- 0.0211")

gold rows: 216

(A) agreement, all 216 rows: aspect 0.9259  sentiment 0.9676
    rows matched in all 3 runs (n=164): aspect 0.9146  sentiment 0.9695
    your reported values: aspect 0.9146, sentiment 0.9695

(B)
      aspectF1_all  aspectF1_matched
run                                
1          0.6934            0.7696
2          0.6583            0.7320
3          0.7067            0.7815
mean
 aspectF1_all        0.6861
aspectF1_matched    0.7611
sd (n-1)
 aspectF1_all        0.0250
aspectF1_matched    0.0258
sd (n)
 aspectF1_all        0.0204
aspectF1_matched    0.0211
    your reported value: 0.7610 +- 0.0211


In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import f1_score

GOLD_COL = 'sentiment'        # <- gold sentiment column in llm_run_N_merged.csv
PRED_COL = 'llm_sentiment'    # <- LLM sentiment column (check df.columns)
labels = ['positive', 'neutral', 'negative']

def sent_f1_all(path):
    df = pd.read_csv(path)

    # keep only gold rows (an outer merge also holds LLM-only extra rows)
    df = df[df[GOLD_COL].notna()].copy()

    # matched = the LLM produced a prediction for this gold row (compute BEFORE astype(str))
    matched = df[PRED_COL].notna()

    y_true = df[GOLD_COL].astype(str).str.strip().str.lower()
    y_pred = df[PRED_COL].astype(str).str.strip().str.lower()
    y_pred[~matched] = 'unmatched'          # not one of the 3 labels -> always wrong

    print(path, '| gold rows:', len(df), '| matched:', int(matched.sum()))
    return f1_score(y_true, y_pred, labels=labels, average='macro')   # macro over the 3 labels only

scores = [sent_f1_all(f'llm_run_{i}_merged.csv') for i in (1, 2, 3)]
print([round(s, 4) for s in scores])
print('mean:', round(np.mean(scores), 4))
print('sample sd (n-1):', round(np.std(scores, ddof=1), 4))
print('population sd (n):', round(np.std(scores, ddof=0), 4))

llm_run_1_merged.csv | gold rows: 216 | matched: 165
llm_run_2_merged.csv | gold rows: 216 | matched: 165
llm_run_3_merged.csv | gold rows: 216 | matched: 165
[0.7445, 0.7431, 0.7495]
mean: 0.7457
sample sd (n-1): 0.0034
population sd (n): 0.0028
